#1. USING SENTIMENT140 Dataset


In [ ]:

import re
import numpy as np
import pandas as pd

# Text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML + evaluation
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Semantic embeddings
from sentence_transformers import SentenceTransformer, util

# Saving models
import joblib

In [ ]:
DATA_PATH = "training.1600000.processed.noemoticon.csv"  # change to your actual CSV path
N_SAMPLES = 50000
RANDOM_STATE = 42

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))
# Add some Twitter-specific stopwords
stop_words.update(["rt", "via"])

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
def clean_tweet(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+", " ", text)   # mentions
    text = re.sub(r"#\w+", " ", text)   # hashtags

    # Remove numbers and punctuation (keep letters + spaces)
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize + stopwords + lemmatization
    tokens = []
    for token in text.split():
        if token in stop_words:
            continue
        lemma = lemmatizer.lemmatize(token)
        tokens.append(lemma)

    return " ".join(tokens)

In [ ]:
def load_sentiment140(path: str, n_samples: int = 8000, random_state: int = 42):
    cols = ["target", "id", "date", "flag", "user", "text"]
    df = pd.read_csv(path, encoding="latin-1", header=None, names=cols)

    df = df[["target", "text"]]
    df = df[df["target"].isin([0, 4])]
    df["label"] = df["target"].map({0: 0, 4: 1})
    df.drop(columns=["target"], inplace=True)

    # ✅ stratified sampling FIX
    df_sampled, _ = train_test_split(
        df,
        train_size=n_samples,
        stratify=df["label"],
        random_state=random_state
    )
    df_sampled = df_sampled.reset_index(drop=True)

    # cleaning
    df_sampled["clean_text"] = df_sampled["text"].apply(clean_tweet)
    df_sampled = df_sampled[df_sampled["clean_text"].str.strip() != ""].reset_index(drop=True)

    return df_sampled


In [ ]:

# -------------------------------
# 4. Train / test split
# -------------------------------

def train_test_split_data(df):
    X = df["clean_text"].values
    y = df["label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test

In [ ]:
# 5. Baseline: TF-IDF + Logistic Regression
# (for comparison only, NOT saved)
# -------------------------------

def train_tfidf_logreg(X_train, X_test, y_train, y_test):
    print("=== Baseline: TF-IDF + Logistic Regression ===")

    tfidf = TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
    )

    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    logreg = LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    logreg.fit(X_train_tfidf, y_train)

    y_pred = logreg.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}\n")

    # We RETURN metrics only; no saving
    return (acc, prec, rec, f1)

In [ ]:
def train_sbert_logreg(X_train, X_test, y_train, y_test):
    print("=== Semantic: SBERT + Logistic Regression (with Hyperparameter Tuning) ===")

    # Load SBERT model
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    sbert = SentenceTransformer(model_name)

    # Encode training and testing text
    X_train_emb = sbert.encode(
        list(X_train),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    X_test_emb = sbert.encode(
        list(X_test),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    # ------------------------------
    # Hyperparameter Tuning
    # ------------------------------
    from sklearn.model_selection import GridSearchCV
    from sklearn.linear_model import LogisticRegression

    param_grid = {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs", "liblinear"],
        "class_weight": [None, "balanced"],
        "penalty": ["l2"],
        "max_iter": [1000],
    }

    print("\n🔍 Running GridSearchCV for Logistic Regression...")
    logreg = LogisticRegression(random_state=RANDOM_STATE)

    grid = GridSearchCV(
        estimator=logreg,
        param_grid=param_grid,
        cv=3,
        scoring="f1",
        n_jobs=-1,
        verbose=1,
    )

    grid.fit(X_train_emb, y_train)

    # Best model from tuning
    best_logreg = grid.best_estimator_
    print("\n✅ Best Hyperparameters:", grid.best_params_)

    # ------------------------------
    # Evaluate the tuned model
    # ------------------------------
    y_pred = best_logreg.predict(X_test_emb)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print("\n📊 SBERT + Logistic Regression Performance")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}\n")

    return sbert, best_logreg, (acc, prec, rec, f1)


In [ ]:
# -------------------------------
# 7. Cosine similarity demo
# -------------------------------

def demo_cosine_similarity(sbert):
    print("=== Demo: Cosine similarity with SBERT embeddings ===")

    s1 = "I love this movie, it was fantastic!"
    s2 = "That film was amazing, I really enjoyed it."
    s3 = "The pizza was terrible and I will not order again."

    emb = sbert.encode([s1, s2, s3], convert_to_tensor=True)
    sim_12 = util.cos_sim(emb[0], emb[1]).item()
    sim_13 = util.cos_sim(emb[0], emb[2]).item()

    print(f"Similarity(s1, s2): {sim_12:.4f} (both positive)")
    print(f"Similarity(s1, s3): {sim_13:.4f} (positive vs negative)\n")

In [ ]:
def save_sbert_model(sbert_logreg):
    joblib.dump(sbert_logreg, "sbert_logreg.pkl")
    print("✅ Saved: sbert_logreg.pkl (Logistic Regression on SBERT embeddings)\n")


In [ ]:
# -------------------------------
# 9. Main
# -------------------------------

if __name__ == "__main__":
    # Load & preprocess
    df = load_sentiment140(DATA_PATH, n_samples=N_SAMPLES, random_state=RANDOM_STATE)
    print(df.head())
    print(f"\nDataset size after cleaning: {len(df)} rows\n")

    X_train, X_test, y_train, y_test = train_test_split_data(df)

    # Baseline TF-IDF (comparison only)
    tfidf_scores = train_tfidf_logreg(X_train, X_test, y_train, y_test)

    # SBERT model (this is the one used by the app)
    sbert, sbert_logreg, sbert_scores = train_sbert_logreg(X_train, X_test, y_train, y_test)

    # Compare
    print("=== Comparison: TF-IDF vs SBERT ===")
    print("Metric       TF-IDF    SBERT")
    names = ["Accuracy", "Precision", "Recall", "F1-score"]
    for name, b, s in zip(names, tfidf_scores, sbert_scores):
        print(f"{name:<11} {b:7.4f}  {s:7.4f}")
    print()

    # Demo cosine similarity
    demo_cosine_similarity(sbert)

    # Save only SBERT Logistic Regression model
    save_sbert_model(sbert_logreg)

    print("Done.")

                                                text  label  \
0  @AngieMC1966 Oh wow. I am so sorry. That's a r...      0   
1   Tired. Sleepy. Exhausted. Need my beauty sleep!       0   
2                      @Net I wouldn't bet on it...       1   
3        I intend to live forever - so far, so good       1   
4                                    Such bad pains       0   

                                 clean_text  
0  oh wow sorry really tough situation deal  
1  tired sleepy exhausted need beauty sleep  
2                                       bet  
3              intend live forever far good  
4                                  bad pain  

Dataset size after cleaning: 49746 rows

=== Baseline: TF-IDF + Logistic Regression ===
Accuracy  : 0.7536
Precision : 0.7477
Recall    : 0.7649
F1-score  : 0.7562

=== Semantic: SBERT + Logistic Regression (with Hyperparameter Tuning) ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/622 [00:00<?, ?it/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]


🔍 Running GridSearchCV for Logistic Regression...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✅ Best Hyperparameters: {'C': 10, 'class_weight': 'balanced', 'max_iter': 1000, 'penalty': 'l2', 'solver': 'liblinear'}

📊 SBERT + Logistic Regression Performance
Accuracy  : 0.7432
Precision : 0.7379
Recall    : 0.7538
F1-score  : 0.7458

=== Comparison: TF-IDF vs SBERT ===
Metric       TF-IDF    SBERT
Accuracy     0.7536   0.7432
Precision    0.7477   0.7379
Recall       0.7649   0.7538
F1-score     0.7562   0.7458

=== Demo: Cosine similarity with SBERT embeddings ===
Similarity(s1, s2): 0.8037 (both positive)
Similarity(s1, s3): 0.2343 (positive vs negative)

✅ Saved: sbert_logreg.pkl (Logistic Regression on SBERT embeddings)

Done.


#Using IMDB movie tweets dataset


In [ ]:
# Colab / environment setup
!pip install -q datasets sentence-transformers scikit-learn gradio joblib nltk

In [ ]:
# train_hf_imdb.py
import re
import numpy as np
import joblib
import multiprocessing

# HuggingFace datasets
from datasets import load_dataset

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# sentence-transformers
from sentence_transformers import SentenceTransformer

# Config
RANDOM_STATE = 42
SAMPLE_SIZE = None  # set to an int (e.g., 20000) to sample; None uses full HF split
TFIDF_MAX_FEATURES = 20000
SBERT_BATCH_SIZE = 32  # lower if RAM/GPU limited
GRID_CV = 3
N_JOBS = min(8, multiprocessing.cpu_count())

# NLTK setup
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
stop_words = set(stopwords.words("english"))
stop_words.update(["br"])  # sometimes present in scraped reviews
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    s = text.lower()
    s = re.sub(r"<.*?>", " ", s)                 # remove html tags
    s = re.sub(r"http\S+|www\.\S+", " ", s)      # remove urls
    s = re.sub(r"[^a-z\s]", " ", s)              # keep letters & spaces
    s = re.sub(r"\s+", " ", s).strip()
    tokens = []
    for tok in s.split():
        if tok in stop_words:
            continue
        tokens.append(lemmatizer.lemmatize(tok))
    return " ".join(tokens)

def load_and_prepare(sample_size=None):
    ds = load_dataset("imdb")

    train = ds["train"]
    test = ds["test"]

    # Convert HF column objects → Python lists
    train_texts = list(train["text"])
    test_texts = list(test["text"])
    all_texts = train_texts + test_texts

    train_labels = list(train["label"])
    test_labels = list(test["label"])
    all_labels = train_labels + test_labels

    print("Raw dataset size:", len(all_texts))   # should be 50,000

    # Optional sampling
    if sample_size is not None:
        X_sample, _, y_sample, _ = train_test_split(
            all_texts,
            all_labels,
            train_size=sample_size,
            stratify=all_labels,
            random_state=RANDOM_STATE
        )
        texts = X_sample
        labels = y_sample
    else:
        texts = all_texts
        labels = all_labels

    print("Cleaning texts...")
    cleaned_texts = [clean_text(t) for t in texts]

    final_texts = []
    final_labels = []
    for t, l in zip(cleaned_texts, labels):
        if t.strip():
            final_texts.append(t)
            final_labels.append(l)

    print("Final usable samples:", len(final_texts))
    return final_texts, np.array(final_labels)


def train_tfidf_baseline(X_train_text, X_test_text, y_train, y_test):
    print("Training TF-IDF baseline...")
    tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1,2))
    X_tr_tfidf = tfidf.fit_transform(X_train_text)
    X_te_tfidf = tfidf.transform(X_test_text)

    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=N_JOBS)
    clf.fit(X_tr_tfidf, y_train)
    y_pred = clf.predict(X_te_tfidf)
    return tfidf, clf, (
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    )

def train_sbert_logreg(X_train_text, X_test_text, y_train, y_test):
    print("Loading SBERT and encoding...")
    sbert_model_name = "sentence-transformers/all-MiniLM-L6-v2"
    sbert = SentenceTransformer(sbert_model_name)

    X_train_emb = sbert.encode(list(X_train_text), batch_size=SBERT_BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
    X_test_emb = sbert.encode(list(X_test_text), batch_size=SBERT_BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)

    print("Running GridSearchCV for LogisticRegression...")
    param_grid = {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs", "liblinear"],
        "class_weight": [None, "balanced"],
        "penalty": ["l2"],
        "max_iter": [1000],
    }

    base = LogisticRegression(random_state=RANDOM_STATE)
    grid = GridSearchCV(base, param_grid=param_grid, cv=GRID_CV, scoring="f1", n_jobs=N_JOBS, verbose=1)
    grid.fit(X_train_emb, y_train)

    best = grid.best_estimator_
    print("Best params:", grid.best_params_)

    y_pred = best.predict(X_test_emb)
    scores = (
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    )
    return sbert, best, scores

def main():
    # load & clean (set SAMPLE_SIZE to limit)
    X, y = load_and_prepare(sample_size=SAMPLE_SIZE)

    # train/test split (stratified)
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    print("Train size:", len(X_train_text), "Test size:", len(X_test_text))

    # Baseline
    tfidf, tfidf_clf, tfidf_scores = train_tfidf_baseline(X_train_text, X_test_text, y_train, y_test)
    print("TF-IDF scores (Acc,Prec,Rec,F1):", tfidf_scores)

    # SBERT
    sbert, sbert_clf, sbert_scores = train_sbert_logreg(X_train_text, X_test_text, y_train, y_test)
    print("SBERT scores (Acc,Prec,Rec,F1):", sbert_scores)

    # Compare
    print("=== Comparison TF-IDF vs SBERT ===")
    names = ["Accuracy", "Precision", "Recall", "F1"]
    for name, t, s in zip(names, tfidf_scores, sbert_scores):
        print(f"{name:<10} TF-IDF={t:.4f}  SBERT={s:.4f}")

    # Save SBERT classifier only
    joblib.dump({"clf": sbert_clf, "scaler": None}, "sbert_logreg2.pkl")
    print("Saved SBERT classifier -> sbert_logreg2.pkl")

# Save TF-IDF vectorizer + classifier together (recommended)
    joblib.dump({"vectorizer": tfidf, "clf": tfidf_clf}, "tfidf_logreg.pkl")
    print("Saved TF-IDF vectorizer+clf -> tfidf_logreg.pkl")

# Optional: also save vectorizer alone for inspection/reuse
# joblib.dump(tfidf, "tfidf_vectorizer.joblib")
# print("Saved TF-IDF vectorizer -> tfidf_vectorizer.joblib")

if __name__ == "__main__":
    main()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Running GridSearchCV for LogisticRegression...
Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best params: {'C': 10, 'class_weight': None, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'liblinear'}
SBERT scores (Acc,Prec,Rec,F1): (0.8266, 0.8266, 0.8266, 0.8266)
=== Comparison TF-IDF vs SBERT ===
Accuracy   TF-IDF=0.8988  SBERT=0.8266
Precision  TF-IDF=0.8891  SBERT=0.8266
Recall     TF-IDF=0.9112  SBERT=0.8266
F1         TF-IDF=0.9000  SBERT=0.8266
Saved SBERT classifier -> sbert_logreg2.pkl
Saved TF-IDF vectorizer+clf -> tfidf_logreg.pkl


In [ ]:
pip install gradio==4.31.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.9/315.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: tomlkit
    Found existing installation: tomlkit 0.13.3
    Uninstalling tomlkit-0.13.3:
      Successfully uninstalled tomlkit-0.13.3
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: numpy
    Found

# RUN THE CELL BLOCK BELOW TO START OUR SIMPLE GRADIO INTERFACE APPLICATION AFTER INSTALLING THE DEPENDENCIES


In [ ]:
# Robust Gradio app with detailed error logging (run this cell)
import os, re, joblib, traceback
import numpy as np
import gradio as gr
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# NLTK setup
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
stop_words = set(stopwords.words("english"))
stop_words.update(["rt","via"])
lemmatizer = WordNetLemmatizer()

def clean_tweet(text: str) -> str:
    try:
        if not isinstance(text, str): return ""
        t = text.lower()
        t = re.sub(r"http\S+|www\.\S+", " ", t)
        t = re.sub(r"@\w+", " ", t)
        t = re.sub(r"#\w+", " ", t)
        t = re.sub(r"[^a-z\s]", " ", t)
        t = re.sub(r"\s+", " ", t).strip()
        return " ".join(lemmatizer.lemmatize(tok) for tok in t.split() if tok not in stop_words)
    except Exception:
        print("Error in clean_tweet:")
        traceback.print_exc()
        return ""

# --- Load SBERT & classifier
print("Loading SBERT model...")
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sbert_clf = None
sbert_scaler = None
try:
    s = joblib.load("sbert_logreg2.pkl")
    print("Loaded sbert_logreg2.pkl type:", type(s))
    if isinstance(s, dict):
        sbert_clf = s.get("clf") or s.get("classifier") or s.get("logreg") or s.get("model")
        sbert_scaler = s.get("scaler", None)
    else:
        # classifier or pipeline object
        if hasattr(s, "predict") or hasattr(s, "predict_proba"):
            sbert_clf = s
    print("sbert_clf:", type(sbert_clf), " sbert_scaler:", type(sbert_scaler))
except Exception as e:
    print("Failed to load sbert_logreg2.pkl:", repr(e))
    traceback.print_exc()

# --- Load TF-IDF model (vectorizer+clf or pipeline)
tfidf_clf = None
tfidf_vec = None
tfidf_pipeline = None
tfidf_ok = False
try:
    tf = joblib.load("tfidf_logreg.pkl")
    print("Loaded tfidf_logreg.pkl type:", type(tf))
    if isinstance(tf, dict):
        # dict might contain vectorizer+clf
        for k in ("vectorizer","tfidf","vect","vectoriser","tfidf_vect","vect0"):
            if k in tf: tfidf_vec = tf[k]
        for k in ("clf","classifier","logreg","model"):
            if k in tf: tfidf_clf = tf[k]
    elif hasattr(tf, "predict_proba") and hasattr(tf, "transform"):
        # an object that can both transform and predict_proba
        tfidf_pipeline = tf
    elif isinstance(tf, (list, tuple)) and len(tf) >= 2:
        tfidf_vec, tfidf_clf = tf[0], tf[1]
    else:
        # try heuristics
        if hasattr(tf, "transform"):
            tfidf_vec = tf
        if hasattr(tf, "predict_proba"):
            tfidf_clf = tf
    print("tfidf_vec:", type(tfidf_vec), " tfidf_clf:", type(tfidf_clf), " pipeline:", type(tfidf_pipeline))
    if tfidf_clf is not None or tfidf_pipeline is not None:
        tfidf_ok = True
except FileNotFoundError:
    print("tfidf_logreg.pkl not found in cwd:", os.getcwd())
except Exception as e:
    print("Error loading tfidf_logreg.pkl:", repr(e))
    traceback.print_exc()

# Utility to run predict_proba robustly
def safe_predict_proba(clf, X):
    """Return a probability vector [neg, pos] or raise informative error."""
    if clf is None:
        raise ValueError("Classifier is None")
    # classifier pipeline that expects raw text
    try:
        if hasattr(clf, "predict_proba"):
            return clf.predict_proba(X)
        # some pipelines require 2D array for text too
        if hasattr(clf, "predict"):
            pred = clf.predict(X)
            # convert to dummy probabilities
            probs = []
            for p in pred:
                probs.append([1.0-p, p] if isinstance(p, (int, np.integer)) else [0.0, 1.0])
            return np.array(probs)
        raise ValueError("Classifier lacks predict_proba or predict")
    except Exception:
        print("Error inside safe_predict_proba:")
        traceback.print_exc()
        raise

# Prediction functions with error handling
def predict_sbert(text):
    try:
        if sbert_clf is None:
            return ("SBERT model not loaded", 0.0, [0.0, 0.0])
        clean = clean_tweet(text)
        if clean == "":
            return ("Input empty after cleaning", 0.0, [0.0,0.0])
        emb = sbert.encode([clean], convert_to_numpy=True)
        if sbert_scaler is not None:
            emb = sbert_scaler.transform(emb)
        proba = safe_predict_proba(sbert_clf, emb)[0]
        pred = int(np.argmax(proba))
        label = "Positive 😊" if pred == 1 else "Negative 😞"
        return label, float(np.max(proba)), [float(p) for p in proba]
    except Exception as e:
        print("Exception in predict_sbert():", repr(e))
        traceback.print_exc()
        return ("SBERT error: " + str(e), 0.0, [0.0,0.0])

def predict_tfidf(text):
    try:
        if not tfidf_ok:
            return ("TF-IDF model not loaded", 0.0, [0.0,0.0])
        clean = clean_tweet(text)
        if clean == "":
            return ("Input empty after cleaning", 0.0, [0.0,0.0])
        # pipeline case (handles transform internally)
        if tfidf_pipeline is not None:
            proba = safe_predict_proba(tfidf_pipeline, [clean])[0]
        else:
            # need a vectorizer + classifier
            if tfidf_vec is None:
                # classifier might accept raw text
                proba = safe_predict_proba(tfidf_clf, [clean])[0]
            else:
                vec = tfidf_vec.transform([clean])
                proba = safe_predict_proba(tfidf_clf, vec)[0]
        pred = int(np.argmax(proba))
        label = "Positive 😊" if pred == 1 else "Negative 😞"
        return label, float(np.max(proba)), [float(p) for p in proba]
    except Exception as e:
        print("Exception in predict_tfidf():", repr(e))
        traceback.print_exc()
        return ("TF-IDF error: " + str(e), 0.0, [0.0,0.0])

# Combined function to return side-by-side values so Gradio fields don't crash
def predict_both(text):
    s_label, s_conf, s_probs = predict_sbert(text)
    t_label, t_conf, t_probs = predict_tfidf(text)
    # return as individual outputs compatible with Gradio boxes
    return s_label, round(s_conf,4), str([round(p,4) for p in s_probs]), t_label, round(t_conf,4), str([round(p,4) for p in t_probs])

# Build UI
with gr.Blocks() as demo:
    gr.Markdown("# SBERT vs TF-IDF Sentiment (robust)")
    inp = gr.Textbox(label="Enter text", lines=3, value="I hate this movie, it sucks.")
    btn = gr.Button("Analyze")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### SBERT result")
            s_label_box = gr.Textbox(label="Label (SBERT)")
            s_conf_box = gr.Number(label="Confidence (SBERT)")
            s_probs_box = gr.Textbox(label="Probabilities (SBERT)")
        with gr.Column():
            gr.Markdown("### TF-IDF result")
            t_label_box = gr.Textbox(label="Label (TF-IDF)")
            t_conf_box = gr.Number(label="Confidence (TF-IDF)")
            t_probs_box = gr.Textbox(label="Probabilities (TF-IDF)")
    btn.click(predict_both, inputs=inp, outputs=[s_label_box, s_conf_box, s_probs_box, t_label_box, t_conf_box, t_probs_box])

print("Launching Gradio...")
demo.launch(share=True)


Loading SBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded sbert_logreg2.pkl type: <class 'dict'>
sbert_clf: <class 'sklearn.linear_model._logistic.LogisticRegression'>  sbert_scaler: <class 'NoneType'>
Loaded tfidf_logreg.pkl type: <class 'dict'>
tfidf_vec: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>  tfidf_clf: <class 'sklearn.linear_model._logistic.LogisticRegression'>  pipeline: <class 'NoneType'>
Launching Gradio...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://079fcad6b9db4bbf0b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
